# Лабораторная работа: Площадь под графиком и управление порогом

**Цель работы:** научиться применять кривые PR и ROC для оценки моделей классификации и использовать их для выбора оптимального порога классификации.

**Выполнил(а):** _[Федорович Ксенмя]_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy import argmax

from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, auc,
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay
)

# опционально: yellowbrick
try:
    from yellowbrick.classifier import ROCAUC, PrecisionRecallCurve
    YB_OK = True
except ImportError:
    YB_OK = False
    print("yellowbrick не установлен. Установите: !pip install yellowbrick")

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)

In [ ]:
# 1.1 Генерация датасета
X, y = make_classification(n_samples=1000, n_classes=2, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=2)

# 1.2 Модель логистической регрессии
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 1.3 ROC-кривая
fpr, tpr, roc_thr = roc_curve(y_test, y_pred_proba)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.plot(fpr, tpr, marker='.')
plt.title('ROC-кривая (LogisticRegression)')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.show()

print("ROC-AUC =", roc_auc_score(y_test, y_pred_proba))

# 1.4 PR-кривая
precision, recall, pr_thr = precision_recall_curve(y_test, y_pred_proba)
no_skill = len(y[y == 1]) / len(y)
plt.plot([0, 1], [no_skill, no_skill], linestyle='--')
plt.plot(recall, precision, marker='.')
plt.title('PR-кривая (LogisticRegression)')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.show()

print("PR-AUC =", auc(recall, precision))

# 1.5 Отчёт по умолчанию
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
model2 = KNeighborsClassifier(n_neighbors=250).fit(X_train, y_train)
y_pred2_proba = model2.predict_proba(X_test)[:, 1]

fpr2, tpr2, _ = roc_curve(y_test, y_pred2_proba)
plt.plot([0, 1], [0, 1], linestyle='--', label='Случайный')
plt.plot(fpr,  tpr,  marker='.', label='LogReg')
plt.plot(fpr2, tpr2, marker='.', label='KNN')
plt.legend(); plt.title('Сравнение ROC'); plt.show()

print("ROC-AUC LogReg:", roc_auc_score(y_test, y_pred_proba))
print("ROC-AUC KNN:   ", roc_auc_score(y_test, y_pred2_proba))

p2, r2, _ = precision_recall_curve(y_test, y_pred2_proba)
print("PR-AUC LogReg:", auc(recall, precision))
print("PR-AUC KNN:   ", auc(r2, p2))

In [ ]:
X, y = make_classification(n_samples=1000, n_classes=2,
                           weights=[0.99, 0.01], random_state=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=4)

print('Датасет:  Class0=%d, Class1=%d' % (sum(y==0), sum(y==1)))
print('Обучающая:Class0=%d, Class1=%d' % (sum(y_train==0), sum(y_train==1)))
print('Тестовая: Class0=%d, Class1=%d' % (sum(y_test==0), sum(y_test==1)))

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred_proba = model.predict_proba(X_test)[:, 1]
print(classification_report(model.predict(X_test)*0 + model.predict(X_test), y_test)
      if False else classification_report(y_test, model.predict(X_test)))

# ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.plot(fpr, tpr, marker='.')
plt.title('ROC при дисбалансе'); plt.show()
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

# PR
precision, recall, pr_thr = precision_recall_curve(y_test, y_pred_proba)
no_skill = len(y[y == 1]) / len(y)
plt.plot([0, 1], [no_skill, no_skill], linestyle='--')
plt.plot(recall, precision, marker='.')
plt.title('PR при дисбалансе'); plt.show()
print("PR-AUC:", auc(recall, precision))

In [ ]:
X, y = make_classification(n_samples=10000, n_classes=2,
                           weights=[0.99, 0.01], random_state=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred_proba = model.predict_proba(X_test)[:, 1]

p, r, pr_thr = precision_recall_curve(y_test, y_pred_proba)

f1 = (2 * p * r) / (p + r + 1e-12)
ix = argmax(f1)
best_thr = pr_thr[ix]
print('Порог=%f, F1=%.3f' % (best_thr, f1[ix]))

no_skill = len(y[y == 1]) / len(y)
plt.plot([0, 1], [no_skill, no_skill], linestyle='--')
plt.plot(r, p, marker='.')
plt.scatter(r[ix], p[ix], marker='o', color='black', zorder=5)
plt.title('PR-кривая с оптимумом F1'); plt.show()

# Порог по умолчанию
y_default = model.predict(X_test)
print("=== Порог по умолчанию (0.5) ===")
print(classification_report(y_test, y_default))

# Лучший порог
y_best = (y_pred_proba > best_thr).astype(int)
print("=== Лучший порог (%.3f) ===" % best_thr)
print(classification_report(y_test, y_best))

Задание 1

In [ ]:
# Двумерный датасет
X2, y2 = make_classification(
    n_samples=1000, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.2, random_state=1)

X_tr, X_te, y_tr, y_te = train_test_split(X2, y2, test_size=0.3, random_state=2)
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

# Сетка для границы решений
xx, yy = np.meshgrid(
    np.linspace(X2[:,0].min()-1, X2[:,0].max()+1, 300),
    np.linspace(X2[:,1].min()-1, X2[:,1].max()+1, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_proba = clf.predict_proba(grid)[:, 1].reshape(xx.shape)

thresholds = [0.2, 0.5, 0.8]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, t in zip(axes, thresholds):
    ax.contourf(xx, yy, grid_proba, levels=[0, t, 1],
                colors=['#ffd6d6', '#d6e0ff'], alpha=0.8)
    ax.contour(xx, yy, grid_proba, levels=[t], colors='k', linewidths=1.5)
    ax.scatter(X_te[y_te==0,0], X_te[y_te==0,1], s=12, c='red', label='0')
    ax.scatter(X_te[y_te==1,0], X_te[y_te==1,1], s=12, c='blue', label='1')
    f1 = f1_score(y_te, (proba > t).astype(int))
    ax.set_title(f'Порог={t}, F1={f1:.3f}')
    ax.legend()
plt.tight_layout(); plt.show()

Задание 2

In [ ]:
data = load_breast_cancer()
Xr, yr = data.data, data.target  # 0=malignant, 1=benign
X_tr, X_te, y_tr, y_te = train_test_split(
    Xr, yr, test_size=0.3, stratify=yr, random_state=42)

clf = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

# ROC
fpr, tpr, _ = roc_curve(y_te, proba)
roc_auc = roc_auc_score(y_te, proba)
plt.plot([0,1],[0,1],'--')
plt.plot(fpr, tpr)
plt.title(f'ROC (breast_cancer), AUC={roc_auc:.4f}')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.show()

# PR
p, r, _ = precision_recall_curve(y_te, proba)
pr_auc = auc(r, p)
no_skill = (y_te==1).mean()
plt.plot([0,1],[no_skill,no_skill],'--')
plt.plot(r, p)
plt.title(f'PR (breast_cancer), AUC={pr_auc:.4f}')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.show()

print(classification_report(y_te, clf.predict(X_te),
                            target_names=data.target_names))
print(confusion_matrix(y_te, clf.predict(X_te)))

Датасет breast_cancer хорошо сбалансирован, модель логистической
регрессии показывает высокие ROC-AUC и PR-AUC. Обе кривые лежат
существенно выше базовых линий, что говорит о высоком качестве
классификатора. PR-кривая и ROC-кривая дают согласованные оценки.

Задание 3

In [ ]:
# Продолжаем с breast_cancer
fpr, tpr, roc_thr = roc_curve(y_te, proba)

# gmean = sqrt(TPR * (1 - FPR)) — баланс между чувствительностью и специфичностью
gmean = np.sqrt(tpr * (1 - fpr))
ix = argmax(gmean)
best_thr_roc = roc_thr[ix]
print(f"Оптимальный порог (ROC/gmean): {best_thr_roc:.4f}, "
      f"gmean={gmean[ix]:.4f}, TPR={tpr[ix]:.4f}, FPR={fpr[ix]:.4f}")

plt.plot(fpr, tpr, marker='.')
plt.plot([0,1],[0,1],'--')
plt.scatter(fpr[ix], tpr[ix], color='black', s=80, zorder=5,
            label=f'Порог={best_thr_roc:.3f}')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend()
plt.title('Оптимизация порога по gmean(TPR, 1-FPR)'); plt.show()

y_best_roc = (proba > best_thr_roc).astype(int)
print("\n=== Отчёт при пороге по умолчанию 0.5 ===")
print(classification_report(y_te, clf.predict(X_te), target_names=data.target_names))

print("\n=== Отчёт при оптимальном (по ROC) пороге ===")
print(classification_report(y_te, y_best_roc, target_names=data.target_names))

Задание 4

In [ ]:
# F-beta: beta > 1 => приоритет recall;  beta < 1 => приоритет precision
def fbeta(p, r, beta):
    b2 = beta**2
    return (1 + b2) * p * r / (b2 * p + r + 1e-12)

p, r, pr_thr = precision_recall_curve(y_te, proba)

# Индексы массивов p и r длиннее на 1, чем pr_thr (последняя точка = полный recall)
p_, r_, thr_ = p[:-1], r[:-1], pr_thr

fb_prec = fbeta(p_, r_, beta=0.5)   # приоритет precision
fb_rec  = fbeta(p_, r_, beta=2.0)   # приоритет recall

ix_prec = argmax(fb_prec)
ix_rec  = argmax(fb_rec)
thr_prec = thr_[ix_prec]
thr_rec  = thr_[ix_rec]

print(f"Порог (F0.5, precision): {thr_prec:.4f}, F0.5={fb_prec[ix_prec]:.4f}")
print(f"Порог (F2,   recall):   {thr_rec:.4f}, F2={fb_rec[ix_rec]:.4f}")

# График
plt.plot(r, p, marker='.')
plt.scatter(r_[ix_prec], p_[ix_prec], marker='s', color='green',
            s=100, label=f'F0.5, thr={thr_prec:.3f}')
plt.scatter(r_[ix_rec],  p_[ix_rec],  marker='o', color='red',
            s=100, label=f'F2,   thr={thr_rec:.3f}')
plt.scatter(r, p, s=0)  # рамка
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.legend(); plt.title('Разные пороги для разных приоритетов'); plt.show()

for name, thr in [('F0.5 (precision)', thr_prec), ('F2 (recall)', thr_rec)]:
    yp = (proba > thr).astype(int)
    print(f"\n=== {name}, порог={thr:.3f} ===")
    print(classification_report(y_te, yp, target_names=data.target_names))

Мера F0.5 (β<1) смещает оптимум в сторону высокого precision — модель
выдаёт меньше ложноположительных срабатываний, но снижает recall.
Мера F2 (β>1) даёт более низкий порог, повышая recall (полноту),
но с уменьшением precision. Выбор порога должен опираться на
прикладные требования: цена пропуска vs цена ложной тревоги.

Задание 5

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
Xi, yi = iris.data, iris.target
n_classes = len(iris.target_names)

X_tr, X_te, y_tr, y_te = train_test_split(
    Xi, yi, test_size=0.3, stratify=yi, random_state=42)

clf_mc = LogisticRegression(max_iter=5000, multi_class='multinomial').fit(X_tr, y_tr)
proba_mc = clf_mc.predict_proba(X_te)

# Бинаризация для ROC/PR (One-vs-Rest)
y_bin = label_binarize(y_te, classes=range(n_classes))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['C0', 'C1', 'C2']

# ROC для каждого класса
for i in range(n_classes):
    fpr_i, tpr_i, _ = roc_curve(y_bin[:, i], proba_mc[:, i])
    auc_i = roc_auc_score(y_bin[:, i], proba_mc[:, i])
    axes[0].plot(fpr_i, tpr_i, color=colors[i],
                 label=f'{iris.target_names[i]} (AUC={auc_i:.3f})')
axes[0].plot([0,1],[0,1],'--', color='k')
axes[0].set_title('ROC OvR (iris)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend()

# PR для каждого класса
for i in range(n_classes):
    p_i, r_i, _ = precision_recall_curve(y_bin[:, i], proba_mc[:, i])
    auc_i = auc(r_i, p_i)
    axes[1].plot(r_i, p_i, color=colors[i],
                 label=f'{iris.target_names[i]} (AUC={auc_i:.3f})')
axes[1].set_title('PR OvR (iris)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout(); plt.show()

# Усреднённые метрики
print("macro ROC-AUC (OvR):",
      roc_auc_score(y_bin, proba_mc, average='macro', multi_class='ovr'))
print("micro ROC-AUC (OvR):",
      roc_auc_score(y_bin, proba_mc, average='micro', multi_class='ovr'))

Для многоклассовой задачи кривые строятся по схеме One-vs-Rest:
для каждого класса отдельно вычисляется вероятность и бинарная
разметка. Метрика AUC усредняется:
- **macro** — среднее по классам (всем классам равный вес);
- **micro** — агрегирует TP/FP/FN по всем классам (учитывает дисбаланс).
Для несбалансированных классов micro и macro могут существенно
отличаться, и это следует учитывать при интерпретации.

Задание 6

In [ ]:
if YB_OK:
    from yellowbrick.classifier import ROCAUC, PrecisionRecallCurve

    fig, ax = plt.subplots(figsize=(8, 6))
    viz = ROCAUC(LogisticRegression(max_iter=5000, multi_class='multinomial'),
                 classes=list(iris.target_names), ax=ax)
    viz.fit(X_tr, y_tr); viz.score(X_te, y_te); viz.show()

    fig, ax = plt.subplots(figsize=(8, 6))
    viz = PrecisionRecallCurve(
        LogisticRegression(max_iter=5000, multi_class='multinomial'),
        classes=list(iris.target_names), ax=ax)
    viz.fit(X_tr, y_tr); viz.score(X_te, y_te); viz.show()
else:
    print("Установите: !pip install yellowbrick")

yellowbrick отображает **несколько кривых** для многоклассовой
задачи — по одной для каждого класса (OvR). Каждая кривая описывает
качество распознавания соответствующего класса как «положительного»,
а все остальные классы — как «отрицательные». Разные AUC
соответствуют разной сложности разделения классов.

## Ответы на контрольные вопросы

### 1. Какие модели можно оценивать с помощью классификационных кривых?

Только вероятностные классификаторы — те, у которых есть метод
`predict_proba` (или `decision_function`, дающий скор-оценки).
ROC-кривая требует упорядочения объектов по степени уверенности,
поэтому модели, возвращающие только метку класса (например, чистый
SVM без `probability=True` или некоторые реализации деревьев),
для построения кривых не подходят.

### 2. PR-кривая vs ROC — когда что лучше?

- **ROC** устойчива к балансу классов, хорошо отражает общее
  качество разделения. Используется, когда классы сбалансированы
  и важны обе ошибки (FP и FN).
- **PR** чувствительна к доле положительного класса, лучше отражает
  качество на **сильно несбалансированных** данных, где интерес
  представляет именно миноритарный (положительный) класс.
  PR-AUC не «обманывает» высокой точностью за счёт мажоритарного
  класса, как это бывает с ROC-AUC при большом дисбалансе.

### 3. Усреднение метрик: micro, macro

- **macro** — вычисляем метрику отдельно по каждому классу и берём
  среднее арифметическое (всем классам одинаковый вес).
- **micro** — сначала агрегируем TP/FP/FN по всем классам и затем
  считаем метрику (каждый объект имеет равный вклад; доминируют
  крупные классы).
- Дополнительно: **weighted** — среднее с весами по поддержке (support).

### 4. Как строятся кривые при множественной классификации?

Метод **One-vs-Rest**: для каждого класса c формируется бинарная
разметка (класс c vs все остальные), строится отдельная ROC/PR-кривая.
Затем кривые либо рисуются отдельно по классам, либо усредняются
(macro/micro) — в т.ч. через интерполяцию. Используется также
**One-vs-One**, но OvR чаще применяется для AUC.

### 5. Почему yellowbrick рисует несколько кривых?

Потому что для многоклассовой задачи каждая кривая соответствует
своему классу, рассматриваемому как «положительный» в схеме OvR.
Разные кривые показывают, насколько легко модель отличает
конкретный класс от остальных.

Доп. задание

In [ ]:
# Сравнение нескольких моделей на breast_cancer
models = {
    'LogReg': LogisticRegression(max_iter=5000),
    'KNN (k=25)': KNeighborsClassifier(n_neighbors=25),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=0),
}

plt.figure(figsize=(8, 6))
for name, m in models.items():
    m.fit(X_tr, y_tr)
    pr = m.predict_proba(X_te)[:, 1]
    f, t, _ = roc_curve(y_te, pr)
    a = roc_auc_score(y_te, pr)
    plt.plot(f, t, label=f'{name} (AUC={a:.3f})')
plt.plot([0,1],[0,1],'--', color='k')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend()
plt.title('Сравнение моделей по ROC'); plt.show()

plt.figure(figsize=(8, 6))
for name, m in models.items():
    pr = m.predict_proba(X_te)[:, 1]
    p_, r_, _ = precision_recall_curve(y_te, pr)
    a = auc(r_, p_)
    plt.plot(r_, p_, label=f'{name} (AUC={a:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend()
plt.title('Сравнение моделей по PR'); plt.show()

## Общий вывод

1. Кривые ROC и PR дополняют друг друга: ROC удобна для сбалансированных
   данных, PR — для несбалансированных.
2. Площади под кривыми (ROC-AUC, PR-AUC) дают числовую сводку качества
   классификатора вне зависимости от конкретного порога.
3. Порог по умолчанию (0.5) редко оптимален. Подбор порога по F1,
   gmean(TPr,1−FPR) или F_β существенно меняет баланс precision/recall.
4. На дисбалансе ROC-AUC может вводить в заблуждение — PR-кривая честнее
   отражает качество.
5. Для многоклассовых задач кривые строятся по схеме OvR; усреднение
   macro/micro отвечает на разные вопросы о качестве.